##### Jupyter Notebook is available at : [Batch Docking](https://github.com/MangalamGSinha/DockSuiteX/tree/main/examples/batch_docking.ipynb)

# Batch Docking Workflow with DockSuiteX

In [10]:
import sys 
import os
sys.path.insert(0, os.path.abspath('..')) 

from docksuitex.batch_docking import BatchProtein, BatchLigand, BatchGridCalculator, BatchVinaDocking, BatchAD4Docking
from docksuitex.utils import fetch_pdb, fetch_sdf, view_molecule, parse_vina_log, parse_ad4_dlg

### Fetch and Prepare Proteins

In [11]:
# Fetch multiple protein structures (1HVR, 2TRX) in parallel
fetch_pdb(pdbid = ["1HVR", "2TRX"], save_to="proteins", parallel=2)

✅ Downloaded 1HVR.pdb → C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\proteins\1HVR.pdb
✅ Downloaded 2TRX.pdb → C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\proteins\2TRX.pdb


[WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/proteins/1HVR.pdb'),
 WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/proteins/2TRX.pdb')]

In [12]:
# Initialize BatchProtein to process all proteins in the 'proteins' directory
# This cleans the PDB, removes water/heterogens, and adds hydrogens/charges

batch_prot = BatchProtein(
    inputs="proteins",

    #Default parameters
    fix_pdb=True,
    remove_heterogens=True,
    remove_water=True,
    add_hydrogens=True,
    add_charges=True,
    preserve_charge_types=None, #eg: ["Zn", "Fe"]
)


# Run preparation for all proteins using 8 CPUs
batch_prot.prepare_all(save_to="prepared_proteins", cpu = 8)

Starting protein preparation for 2 files...
Using 2 parallel workers, 1 CPUs per worker
Output directory: C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\prepared_proteins
✅ Batch processing completed!


[{'file': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\proteins\\1HVR.pdb',
  'status': 'success',
  'pdbqt_path': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\prepared_proteins\\1HVR.pdbqt'},
 {'file': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\proteins\\2TRX.pdb',
  'status': 'success',
  'pdbqt_path': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\prepared_proteins\\2TRX.pdbqt'}]

### Find Binding Pockets

In [13]:
# Identify binding pockets for all prepared proteins using P2Rank
batch_gc = BatchGridCalculator(inputs="prepared_proteins",  max_pockets=3, mode="p2rank")

# Run pocket detection and return receptors with their pocket centers
grid_results = batch_gc.run_all(save_to="p2rank_outputs", cpu=8)

Starting batch grid calculation (p2rank mode)
Using 2 parallel workers, 4 CPUs per worker
Output directory: C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\p2rank_outputs
✅ 1HVR.pdbqt  →  2 pocket(s)
[{'center': (-9.2421, 16.1006, 27.6026),
  'grid_size': (35.032, 33.316, 39.992),
  'probability': 0.929,
  'rank': 1},
 {'center': (-12.5447, 21.7728, 45.5697),
  'grid_size': (20.891, 24.788, 25.599),
  'probability': 0.012,
  'rank': 2}]
✅ 2TRX.pdbqt  →  3 pocket(s)
[{'center': (24.8541, 31.8383, 10.2135),
  'grid_size': (23.651, 22.617, 27.608),
  'probability': 0.132,
  'rank': 1},
 {'center': (12.529, 36.7841, 19.8122),
  'grid_size': (23.751, 23.242, 29.613),
  'probability': 0.105,
  'rank': 2},
 {'center': (25.6872, 31.3306, -7.9765),
  'grid_size': (23.39, 23.791, 28.063),
  'probability': 0.037,
  'rank': 3}]
✅ Batch grid calculation completed.


### 2. Fetch and Prepare Ligands

In [14]:
# Fetch multiple ligand structures in parallel
fetch_sdf(molecule_id=["228", "338"], save_to="ligands", parallel=2)

✅ Downloaded 228.sdf (PubChem) → C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\ligands\228.sdf
✅ Downloaded 338.sdf (PubChem) → C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\ligands\338.sdf


[WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/ligands/228.sdf'),
 WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/ligands/338.sdf')]

In [15]:
# Initialize BatchLigand to process all ligands in the 'ligands' directory
# This minimizes the structure, removes water and adds hydrogens/charges

batch_lig = BatchLigand(
    inputs="ligands",

    #Default parameters
    minimize=None,  #Options: "mmff94", "mmff94s", "uff", "gaff"
    remove_water=True,
    add_hydrogens=True,
    add_charges=True,
    preserve_charge_types=None,
)
# Run preparation for all ligands using 8 CPUs
batch_lig.prepare_all(save_to="prepared_ligands", cpu=8)

Starting ligand preparation for 2 files...
Using 2 parallel workers, 1 CPUs per worker
Output directory: C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\prepared_ligands
✅ Batch processing completed!


[{'file': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\ligands\\338.sdf',
  'status': 'success',
  'pdbqt_path': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\prepared_ligands\\338.pdbqt'},
 {'file': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\ligands\\228.sdf',
  'status': 'success',
  'pdbqt_path': 'C:\\Users\\Mangalam\\Desktop\\New folder (2)\\DockSuiteX\\examples\\prepared_ligands\\228.pdbqt'}]

### Dock each ligand against a each receptor at all the predicted protein pockets

##### Using AutoDock Vina

In [16]:
# Initialize BatchVinaDocking with prepared receptors (and their pockets) and ligands
batch_vina = BatchVinaDocking(
    receptors_with_pockets=grid_results,
    ligands="prepared_ligands",
    seed=42, # Random seed for reproducibility

    # Default Parameters
    exhaustiveness=8,
    num_modes=9, 
)

# Run Vina docking for all receptor-ligand pairs using 8 CPUs
batch_vina.run_all(save_to="vina_batch_results", cpu=8)

Starting AutoDock Vina docking for 10 tasks...
Using 8 parallel workers, 1 CPUs per worker
Output directory: C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results
✅ Batch processing completed!



{('1HVR.pdbqt',
  '338.pdbqt',
  (-12.5447,
   21.7728,
   45.5697)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/1HVR_338_center_-12.54_21.77_45.57'),
 ('2TRX.pdbqt',
  '338.pdbqt',
  (24.8541,
   31.8383,
   10.2135)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/2TRX_338_center_24.85_31.84_10.21'),
 ('2TRX.pdbqt',
  '338.pdbqt',
  (12.529,
   36.7841,
   19.8122)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/2TRX_338_center_12.53_36.78_19.81'),
 ('1HVR.pdbqt',
  '338.pdbqt',
  (-9.2421,
   16.1006,
   27.6026)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/1HVR_338_center_-9.24_16.10_27.60'),
 ('2TRX.pdbqt',
  '338.pdbqt',
  (25.6872,
   31.3306,
   -7.9765)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/2TRX_338_center_25.69_31.33_-7.98'),
 ('1

In [17]:
# Parse all Vina output logs into a single CSV summary
batch_vina.parse_results()

Starting Vina log parsing for 10 file(s)...
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\1HVR_338_center_-12.54_21.77_45.57\log.txt → 9 modes
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\2TRX_338_center_24.85_31.84_10.21\log.txt → 9 modes
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\2TRX_338_center_12.53_36.78_19.81\log.txt → 9 modes
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\1HVR_338_center_-9.24_16.10_27.60\log.txt → 9 modes
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\2TRX_338_center_25.69_31.33_-7.98\log.txt → 9 modes
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\1HVR_228_center_-9.24_16.10_27.60\log.txt → 9 modes
✅ Parsed C:\Users\Mangalam\Desktop\New folder (2)\DockSuiteX\examples\vina_batch_results\1HVR_228_center_-12.

,Receptor,Ligand,Grid Center X,Grid Center Y,Grid Center Z,Grid Size X,Grid Size Y,Grid Size Z,Grid Spacing,Exhaustiveness,Mode,Affinity (kcal/mol),RMSD LB,RMSD UB
0,1HVR,338,-12.5447,21.7728,45.5697,20.891,24.788,25.599,0.375,8,1,-4.794,0.000,0.000
1,1HVR,338,-12.5447,21.7728,45.5697,20.891,24.788,25.599,0.375,8,2,-4.759,1.483,2.811
2,1HVR,338,-12.5447,21.7728,45.5697,20.891,24.788,25.599,0.375,8,3,-4.713,10.160,11.160
3,1HVR,338,-12.5447,21.7728,45.5697,20.891,24.788,25.599,0.375,8,4,-4.442,9.782,11.020
4,1HVR,338,-12.5447,21.7728,45.5697,20.891,24.788,25.599,0.375,8,5,-4.251,9.836,11.120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,2TRX,228,12.5290,36.7841,19.8122,23.751,23.242,29.613,0.375,8,5,-5.130,5.868,8.998
86,2TRX,228,12.5290,36.7841,19.8122,23.751,23.242,29.613,0.375,8,6,-5.014,6.644,8.873
87,2TRX,228,12.5290,36.7841,19.8122,23.751,23.242,29.613,0.375,8,7,-4.955,5.225,8.597
88,2TRX,228,12.5290,36.7841,19.8122,23.751,23.242,29.613,0.375,8,8,-4.945,16.450,17.700


In [18]:
# Generate ProLIF interaction fingerprints for all successful docking results.
# This identifies hydrogen bonds, hydrophobic contacts, etc.
batch_vina.interaction_profile(cpu=2)

Starting ProLIF interaction profiling for 10 results...
Using 2 parallel workers, 1 CPUs per worker
✅ Batch interaction profiling completed!
{('1HVR.pdbqt', '228.pdbqt', (-12.5447, 21.7728, 45.5697)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/1HVR_228_center_-12.54_21.77_45.57/prolif_results'),
 ('1HVR.pdbqt', '228.pdbqt', (-9.2421, 16.1006, 27.6026)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/1HVR_228_center_-9.24_16.10_27.60/prolif_results'),
 ('1HVR.pdbqt', '338.pdbqt', (-12.5447, 21.7728, 45.5697)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/1HVR_338_center_-12.54_21.77_45.57/prolif_results'),
 ('1HVR.pdbqt', '338.pdbqt', (-9.2421, 16.1006, 27.6026)): WindowsPath('C:/Users/Mangalam/Desktop/New folder (2)/DockSuiteX/examples/vina_batch_results/1HVR_338_center_-9.24_16.10_27.60/prolif_results'),
 ('2TRX.pdbqt', '228.pdbqt', (12.529, 3

ligand             Info                                       UNL1             \
protein        Receptor     Ligand             Center Pose ASP15.0              
interaction                                                HBDonor VdWContact   
0            2TRX.pdbqt  338.pdbqt  24.85_31.84_10.21    1   False      False   
1            2TRX.pdbqt  338.pdbqt  24.85_31.84_10.21    2   False      False   
2            2TRX.pdbqt  338.pdbqt  24.85_31.84_10.21    3   False      False   
3            2TRX.pdbqt  338.pdbqt  24.85_31.84_10.21    4   False      False   
4            2TRX.pdbqt  338.pdbqt  24.85_31.84_10.21    5   False      False   
..                  ...        ...                ...  ...     ...        ...   
85           2TRX.pdbqt  228.pdbqt  12.53_36.78_19.81    5   False      False   
86           2TRX.pdbqt  228.pdbqt  12.53_36.78_19.81    6   False      False   
87           2TRX.pdbqt  228.pdbqt  12.53_36.78_19.81    7   False      False   
88           2TRX.pdbqt  228.pdbqt  12.53_36.78_19.81    8   False       True   
89           2TRX.pdbqt  228.pdbqt  12.53_36.78_19.81    9   False      False   

ligand                                                   ...             \
protein        THR54.0    ALA93.1   HOH404.2   HOH412.2  ...    SER11.1   
interaction VdWContact VdWContact VdWContact VdWContact  ... VdWContact   
0                False       True      False      False  ...      False   
1                False      False      False      False  ...      False   
2                 True      False       True      False  ...      False   
3                False       True      False      False  ...      False   
4                False      False      False      False  ...      False   
..                 ...        ...        ...        ...  ...        ...   
85               False      False      False      False  ...      False   
86               False      False      False      False  ...      False   
87               False      False      False      False  ...      False   
88               False      False      False      False  ...      False   
89               False      False      False      False  ...       True   

ligand                                                                      \
protein        TRP28.1 GLU30.1               LYS36.1    ASN63.1   HOH434.3   
interaction VdWContact HBDonor VdWContact VdWContact VdWContact VdWContact   
0                False   False      False      False      False      False   
1                False   False      False      False      False      False   
2                False   False      False      False      False      False   
3                False   False      False      False      False      False   
4                False   False      False      False      False      False   
..                 ...     ...        ...        ...        ...        ...   
85               False   False      False      False      False      False   
86               False   False      False      False      False       True   
87                True    True       True      False       True      False   
88               False   False      False       True      False      False   
89               False   False      False      False      False       True   

ligand                                        
protein       HOH454.3   HOH497.3   HOH505.3  
interaction VdWContact VdWContact VdWContact  
0                False      False      False  
1                False      False      False  
2                False      False      False  
3                False      False      False  
4                False      False      False  
..                 ...        ...        ...  
85               False      False      False  
86                True      False       True  
87                True      False      False  
88               False      False      False  
89               False       True       True  

[90 rows x 130 columns]

##### Using AutoDock4

In [ ]:
# Initialize BatchAD4Docking with prepared receptors (and their pockets) and ligands
batch_ad4 = BatchAD4Docking(
    receptors_with_centers=grid_results,
    ligands="prepared_ligands",
    seed=(21, 42),  #Random seed for reproducibility

    #Default Parameters
    grid_size=(60,60,60),
    spacing=0.375,     
    dielectric=-0.1465, 
    smooth=0.5,          
    ga_pop_size=150,
    ga_num_evals=2500000,
    ga_num_generations=27000,
    ga_elitism=1,
    ga_mutation_rate=0.02,
    ga_crossover_rate=0.8,
    ga_run=10,
    rmstol=2.0,
)

# Run AutoDock4 docking for all receptor-ligand pairs using 8 CPUs
results_ad4 = batch_ad4.run_all(save_to="ad4_batch_results", cpu=8)

In [ ]:
# Parse all AD4 output logs into a single CSV summary
batch_ad4.parse_results()

In [ ]:
# Generate ProLIF interaction fingerprints for all successful docking results.
# This identifies hydrogen bonds, hydrophobic contacts, etc.
batch_ad4.interaction_profile()